# Pet Segmentation & Breed Classification — Live Demo

This notebook runs a **trained model** on any image you provide. **No training happens here** — it only loads a saved model (`best.pth`) and predicts instantly.

**How to use:**
1. Make sure `best.pth` (the trained model) is in the **same folder** as this notebook.
2. Run all cells from top to bottom.
3. When the *Upload* button appears, **drop your image** into it — or set `IMAGE_PATH` below to a file on disk and skip the upload.
4. The last cell shows your image with the predicted pet region overlaid, plus the predicted **breed** and **confidence**.

In [ ]:
from pathlib import Path
from PIL import Image

# Option A: set this to an image path on disk, e.g.
#   IMAGE_PATH = "/home/user/Downloads/my_pet.jpg"
# Leave it empty to use the upload button below instead.
IMAGE_PATH = ""

if IMAGE_PATH.strip():
    _img_path = Path(IMAGE_PATH).expanduser()
    if not _img_path.is_file():
        raise FileNotFoundError(f"image not found: {_img_path}")
else:
    try:
        import ipywidgets as widgets
        _uploader = widgets.FileUpload(
            accept="image/*", multiple=False,
            description="Upload an image",
        )
        display(_uploader)
        print("\nDrop your image into the upload button above, "
              "then re-run this cell and the next one.")
    except ImportError:
        from IPython.display import FileUpload as _IPyUpload
        _uploader = _IPyUpload(accept="image/*", multiple=False)
        display(_uploader)
        print("\nDrop your image into the upload widget above, "
              "then re-run this cell and the next one.")
    if getattr(_uploader, "value", None):
        _uploaded = _uploader.value
        _first = list(_uploaded.values())[0]
        _raw = _first["content"] if isinstance(_first, dict) else _first
        _img_path = Path("./_uploaded_image.png")
        _img_path.write_bytes(bytes(_raw))
        print(f"Loaded uploaded image ({_img_path.stat().st_size} bytes)")

_img = Image.open(_img_path).convert("RGB")
print(f"image: {_img.size[0]} x {_img.size[1]}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):

    def __init__(self, in_ch=3, base_channels=32, num_classes=37):
        super().__init__()
        c = [base_channels, base_channels * 2, base_channels * 4, base_channels * 8]
        self.enc1 = DoubleConv(in_ch, c[0])
        self.enc2 = DoubleConv(c[0], c[1])
        self.enc3 = DoubleConv(c[1], c[2])
        self.enc4 = DoubleConv(c[2], c[3])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(c[3], c[3] * 2)

        self.up4 = nn.ConvTranspose2d(c[3] * 2, c[3], 2, stride=2)
        self.dec4 = DoubleConv(c[3] * 2, c[3])
        self.up3 = nn.ConvTranspose2d(c[3], c[2], 2, stride=2)
        self.dec3 = DoubleConv(c[2] * 2, c[2])
        self.up2 = nn.ConvTranspose2d(c[2], c[1], 2, stride=2)
        self.dec2 = DoubleConv(c[1] * 2, c[1])
        self.up1 = nn.ConvTranspose2d(c[1], c[0], 2, stride=2)
        self.dec1 = DoubleConv(c[0] * 2, c[0])

        self.seg_head = nn.Conv2d(c[0], 1, 1)
        self.cls_head = nn.Sequential(
            nn.Linear(c[3] * 2, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes),
        )

    def encode(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        s4 = self.enc4(self.pool(s3))
        b = self.bottleneck(self.pool(s4))
        return (s1, s2, s3, s4), b

    def decode(self, b, skips):
        s1, s2, s3, s4 = skips
        x = self.dec4(torch.cat([self.up4(b), s4], dim=1))
        x = self.dec3(torch.cat([self.up3(x), s3], dim=1))
        x = self.dec2(torch.cat([self.up2(x), s2], dim=1))
        x = self.dec1(torch.cat([self.up1(x), s1], dim=1))
        return x

    def forward(self, x):
        skips, b = self.encode(x)
        seg_logits = self.seg_head(self.decode(b, skips))
        cls_logits = self.cls_head(F.adaptive_avg_pool2d(b, 1).flatten(1))
        return {"seg_logits": seg_logits, "cls_logits": cls_logits, "bottleneck": b}


import torch
import torch.nn as nn
import torch.nn.functional as F



class AttentionGate(nn.Module):

    def __init__(self, gate_ch, skip_ch, inter_ch):
        super().__init__()
        self.w_g = nn.Sequential(
            nn.Conv2d(gate_ch, inter_ch, 1, bias=False), nn.BatchNorm2d(inter_ch)
        )
        self.w_x = nn.Sequential(
            nn.Conv2d(skip_ch, inter_ch, 1, bias=False), nn.BatchNorm2d(inter_ch)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(inter_ch, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        # The gating signal g is coarser than the skip feature x; upsample it to
        # x's spatial resolution first so the two 1x1 projections can be added.
        g = F.interpolate(
            g, size=x.shape[-2:], mode="bilinear", align_corners=False
        )
        a = self.relu(self.w_g(g) + self.w_x(x))
        return x * self.psi(a)


class AttentionUNet(UNet):

    def __init__(self, in_ch=3, base_channels=32, num_classes=37):
        super().__init__(in_ch=in_ch, base_channels=base_channels, num_classes=num_classes)
        c = [base_channels, base_channels * 2, base_channels * 4, base_channels * 8]
        self.a4 = AttentionGate(c[3] * 2, c[3], c[3] // 2)
        self.a3 = AttentionGate(c[3], c[2], c[2] // 2)
        self.a2 = AttentionGate(c[2], c[1], c[1] // 2)
        self.a1 = AttentionGate(c[1], c[0], c[0] // 2)

    def forward(self, x):
        skips, b = self.encode(x)
        s1, s2, s3, s4 = skips
        d = self.dec4(torch.cat([self.up4(b), self.a4(b, s4)], dim=1))
        d = self.dec3(torch.cat([self.up3(d), self.a3(d, s3)], dim=1))
        d = self.dec2(torch.cat([self.up2(d), self.a2(d, s2)], dim=1))
        d = self.dec1(torch.cat([self.up1(d), self.a1(d, s1)], dim=1))
        seg_logits = self.seg_head(d)
        cls_logits = self.cls_head(F.adaptive_avg_pool2d(b, 1).flatten(1))
        return {"seg_logits": seg_logits, "cls_logits": cls_logits, "bottleneck": b}


__all__ = ["UNet", "AttentionUNet", "build_model", "build_backbone_classifier"]


def build_model(cfg):
    name = cfg["model"]["name"]
    kwargs = dict(
        base_channels=cfg["model"].get("base_channels", 32),
        num_classes=cfg["model"].get("num_classes", NUM_CLASSES),
    )
    if name == "unet":
        return UNet(**kwargs)
    if name == "attention_unet":
        return AttentionUNet(**kwargs)
    raise ValueError(f"unknown model: {name}")


# ---------------------------------------------------------------------------
# Local model factory (self-contained equivalent of the project factory)
# ---------------------------------------------------------------------------

NUM_CLASSES = 37


def build_model(cfg):
    name = cfg["model"]["name"]
    kwargs = dict(
        base_channels=cfg["model"].get("base_channels", 32),
        num_classes=cfg["model"].get("num_classes", NUM_CLASSES),
    )
    if name == "unet":
        return UNet(**kwargs)
    if name == "attention_unet":
        return AttentionUNet(**kwargs)
    raise ValueError(f"unknown model: {name}")


In [ ]:
import torch

# The trained model file. It stores everything needed to rebuild the
# model: architecture name, image size, threshold and the 37 breed
# names. Keep it in the same folder as this notebook.
CKPT_PATH = "best.pth"

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
cfg = ckpt["cfg"]

model = build_model(cfg)
model.load_state_dict(ckpt["model_state"])
model.eval()

CLASSES = list(ckpt.get("classes") or [f"class_{i}" for i in range(37)])
IMG_SIZE = int(cfg.get("data", {}).get("img_size", 256))
THRESHOLD = float(cfg.get("model", {}).get("seg_threshold", 0.5))

print(f"model:        {cfg['model']['name']}")
print(f"image size:  {IMG_SIZE} x {IMG_SIZE}")
print(f"threshold:   {THRESHOLD}")
print(f"breeds:      {len(CLASSES)} classes")
if "epoch" in ckpt:
    print(f"best epoch:  {ckpt['epoch']} "
          f"(val mIoU {ckpt.get('best_miou', float('nan')):.4f})")
print("model loaded — ready to predict.")

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import numpy as np

# Apply exactly the same preprocessing as training:
# resize to IMG_SIZE x IMG_SIZE, then scale pixels to [0, 1].
# (Training used no additional normalization.)
x = TF.resize(_img, [IMG_SIZE, IMG_SIZE], antialias=True)
x = TF.to_tensor(x).unsqueeze(0)

with torch.no_grad():
    out = model(x)

# Segmentation: sigmoid -> binary foreground mask
mask = (torch.sigmoid(out["seg_logits"]) > THRESHOLD).float()
mask = TF.resize(
    mask[0, 0].unsqueeze(0).unsqueeze(0), [_img.size[1], _img.size[0]],
    interpolation=TF.InterpolationMode.NEAREST,
)[0, 0].cpu().numpy()

# Classification: softmax -> breed + confidence
probs = torch.softmax(out["cls_logits"], dim=1)[0]
conf, label = probs.max(0)
breed = CLASSES[int(label)]

print(f"predicted breed:  {breed}")
print(f"confidence:       {conf.item():.2%}")
print(f"foreground:       {100.0 * mask.mean():.1f}% of pixels")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def _overlay(img, mask, alpha=0.5, color=(1.0, 0.0, 0.0)):
    img = np.asarray(img, dtype=np.float32).copy()
    m = np.asarray(mask).astype(bool)
    if m.any():
        img[m] = (1.0 - alpha) * img[m] + alpha * np.asarray(color, dtype=np.float32)
    return np.clip(img, 0.0, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(_img)
axes[0].set_title("Input image")
axes[0].axis("off")
axes[1].imshow(_overlay(_img, mask))
axes[1].set_title(f"Prediction: {breed} ({conf.item():.0%})")
axes[1].axis("off")
plt.tight_layout()
plt.show()

### Running this on a fresh machine

Only two files are needed: this notebook and `best.pth`.
Install the packages with:

```bash
pip install torch torchvision pillow numpy matplotlib ipywidgets
```

Then open this notebook (e.g. `jupyter notebook`) and run it top to bottom. No GPU, dataset, internet, or training code is required.